In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline

# Шаг 1: заполнение пропусков

In [ ]:
# Загружаем данные
df = pd.read_csv('ds14_hotel.csv')

# Признаки с пропусками
numerical_with_nan = ['no_of_weekend_nights', 'no_of_week_nights', 'required_car_parking_space']
X_with_nan = df[numerical_with_nan]

In [ ]:
# Создайте SimpleImputer с fill_value=-999999
imputer = SimpleImputer(fill_value=-999999)

# Обучите imputer на X_with_nan и получите заполненные данные
X_filled = imputer.fit_transform(X_with_nan)

# Шаг 2: кодируем категориальные признаки

In [ ]:
# Обычные категориальные признаки
categorical_features = ['type_of_meal_plan', 'room_type_reserved', 'market_segment_type']
X_categorical = df[categorical_features]

# Бинарный признак
binary_feature = ['is_vip_guest']
X_binary = df[binary_feature]

In [ ]:
# Создайте OneHotEncoder для обычных категориальных признаков
categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Создайте OneHotEncoder для бинарного признака с параметром drop='first'
binary_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

# Обучите и примените оба энкодера
X_categorical_encoded = categorical_encoder.fit_transform(X_categorical)
X_binary_encoded = binary_encoder.fit_transform(X_categorical)

# Шаг 3: работа с выбросами

В данных о бронированиях могут быть выбросы. Например, кто-то забронировал номер на 100 ночей, а большинство гостей останавливается на 2–3 ночи.
Но всё удачно складывается — вы будете работать с деревом решений, и ему выбросы не мешают. Например, если дерево создаёт разбиение no_of_nights <= 7, то значения 8, 50 и 100 попадут в одну ветку — «больше 7 ночей». Точное значение выброса не влияет на предсказание.
Единственное исключение — работа с переобученным деревом. Но в ваших силах не доводить его до такого состояния!
Поэтому в рамках задания ничего делать с выбросами не надо. Дерево само справится.

# Шаг 4: сбор ColumnTransformer

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('binary', binary_encoder, binary_feature),
    ('categorical', categorical_encoder, categorical_features),
    ('imputer', imputer, numerical_with_nan)
], remainder='passthrough')

In [ ]:
# Разделяем на признаки и целевую переменную
X = df.drop(columns=["Booking_ID", "booking_status"])
y = df['booking_status']

In [ ]:
# Разделяем на train и test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Иницилизируйте модель
model = DecisionTreeClassifier(max_depth=10, random_state=42)

In [ ]:
# Создайте Pipeline с preprocessor и DecisionTreeClassifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Обучите pipeline
pipeline.fit(X_train, y_train)

# Получите предсказания (классы)
y_pred = pipeline.predict(X_test)

# Получите вероятности для ROC-AUC
y_proba = pipeline.predict_proba(X_test)

# Рассчитайте метрики
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

# Шаг 5: оценка базовой модели кросс-валидацией

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Оцените модель по Accuracy и ROC-AUC одновременно с помощью кросс-валидации
cv_results = cross_validate(
    estimator=pipeline,
    X, y,
    scoring=['accuracy', 'roc_auc'],
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

In [ ]:
# Выведите результаты
print("Базовая модель (без ограничений) - кросс-валидация на train:")
print(f"Accuracy: {cv_results['test_accuracy'].mean():.3f} (+/- {cv_results['test_accuracy'].std():.3f})")
print(f"ROC-AUC:  {cv_results['test_roc_auc'].mean():.3f} (+/- {cv_results['test_roc_auc'].std():.3f})")
print()
print(f"Accuracy на train: {cv_results['train_accuracy'].mean():.3f} (+/- {cv_results['train_accuracy'].std():.3f})")
print(f"ROC-AUC на train: {cv_results['train_roc_auc'].mean():.3f} (+/- {cv_results['train_roc_auc'].std():.3f})")

In [ ]:
# Посмотрим, насколько глубоко разрослось переобученное дерево. Это подскажет, какие значения гиперпараметров стоит указывать.
overfit_pipeline = pipeline.fit(X_train, y_train)

tree = overfit_pipeline.steps[-1][-1]

print(f"Глубина переобученного дерева: {tree.get_depth()}")
print(f"Количество листьев переобученного дерева: {tree.get_n_leaves()}")

# Шаг 6: подбор гиперпараметров

In [ ]:
# Задайте гиперпараметры для перебора
param_grid = {
    'model__max_depth': [7, 10],
    'model__min_samples_split': [20, 50, 100],
    'model__min_samples_leaf': [10, 50],
    'model__max_leaf_nodes': [50, 100, 500, None]
}

# Инициализируйте GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

# Обучите grid_search
grid_search.fit(X_train, y_train)

print("\nЛучшая комбинация параметров:")
print(grid_search.best_params_) # Выведите лучшую комбинацию
print(f"Лучшее значение ROC-AUC: {grid_search.best_score_:.3f}")

# Шаг 7: обучение лучшей модели

In [ ]:
# Создайте финальный пайплайн с лучшими параметрами
final_pipeline = grid_search.best_estimator_

# Обучите финальный пайплайн
final_pipeline.fit(X_train, y_train)

# Получите предсказания и вероятности
y_test_pred = final_pipeline.predict(X_test)
y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]

# Посчитайте метрики
accuracy_test = accuracy_score(y_test, y_test_pred)
roc_auc_test = roc_auc_score(y_test, y_test_proba)

# Выведите результаты
print("Модель с оптимальными параметрами:")
print(f"Accuracy test:  {accuracy_test:.3f}")
print(f"ROC-AUC test:   {roc_auc_test:.3f}")

In [ ]:
# Оцениваем модель по Accuracy и ROC-AUC одновременно
cv_results = cross_validate(final_pipeline, X_train, y_train, cv=cv,
                            scoring=['accuracy', 'roc_auc'], return_train_score=True)



# Выводим результаты
print("Улучшенная модель - кросс-валидация на test:")
print(f"Accuracy: {cv_results['test_accuracy'].mean():.3f} (+/- {cv_results['test_accuracy'].std():.3f})")
print(f"ROC-AUC:  {cv_results['test_roc_auc'].mean():.3f} (+/- {cv_results['test_roc_auc'].std():.3f})")
print()
print(f"Accuracy на train: {cv_results['train_accuracy'].mean():.3f} (+/- {cv_results['train_accuracy'].std():.3f})")
print(f"ROC-AUC на train: {cv_results['train_roc_auc'].mean():.3f} (+/- {cv_results['train_roc_auc'].std():.3f})")